### HyDE
🧠 What is HyDe?

HyDE (Hypothetical Document Embeddings) is a retrieval technique where, instead of embedding the user’s query directly, you first generate a hypothetical answer (document) to the query using an LLM — and then embed that hypothetical document to search your vector store.

➡️ HyDE bridges the gap between user intent and relevant content, especially when:

1. Queries are short
2. Language mismatch between query and documents
3.You want to retrieve based on answer content, not question words

In [17]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.vectorstores import Chroma

Loading and Chunking the documents

In [18]:
chunk_size=300
chunk_overlap=100

loader=WikipediaLoader(query="Steve Jobs",load_max_docs=5)
documents=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)

docs=text_splitter.split_documents(documents=documents)
docs


[Document(metadata={'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. with his early business partner Steve Wozniak as Apple Computer Company in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was born in San Francisco in 1955 and adopted shortly afterward. He attended Reed College in 1972 before withdrawing that same year. In 1974, he traveled through India, seeking enlightenment before later studying Zen Buddhism. He and Wozniak co-founded Apple in 1976 to further develop and s

In [19]:
from langchain_classic.vectorstores import  FAISS
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(docs,embeddings)

In [20]:
from langchain_classic.chat_models import  init_chat_model
llm = init_chat_model("ollama:llama3.2:latest")
llm.invoke("hi")

AIMessage(content='How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-04-26T09:59:13.5984732Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4977534100, 'load_duration': 4164428000, 'prompt_eval_count': 26, 'prompt_eval_duration': 441208300, 'eval_count': 8, 'eval_duration': 354104800, 'model_name': 'llama3.2:latest'}, id='lc_run--019dc93a-844b-74b0-b068-a70d3851a850-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 8, 'total_tokens': 34})

In [21]:
from langchain_classic.vectorstores import  Chroma
db=Chroma.from_documents(documents=docs,embedding=embeddings,persist_directory="output/steve_jobs")
db.persist()
base_retriever=db.as_retriever(search_kwargs={"k":5})


In [22]:
from langchain_core.output_parsers import  StrOutputParser

# ! prompt
from langchain_classic.prompts.chat import SystemMessagePromptTemplate, ChatPromptTemplate

def get_hyde_doc(query):
    template = """Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic"""

    system_message_prompt = SystemMessagePromptTemplate.from_template(template = template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_prompt(query = query).to_messages()
    # print("messages",messages)
    response = llm.invoke(messages[0].content)
    print(response)
    hypo_doc = response.content
    # print("hypo_doc",hypo_doc)
    return hypo_doc

In [23]:
query="When was Steve Jobs fired from Apple?"
print(get_hyde_doc(query=query))

content="**The Turbulent Departure of Steve Jobs: A Hypothetical Account**\n\nIn our imaginary narrative, Steve Jobs was not actually fired from Apple, but rather, his tumultuous relationship with the company came to an abrupt end due to a combination of factors.\n\nIt was the year 1996, and Apple had just released the Newton PDA, a personal digital assistant that was meant to revolutionize the way people interacted with technology. However, despite Jobs' enthusiasm for the project, it failed to gain significant market traction, leading to a decline in sales and revenue.\n\nAs a result, Apple's board of directors began to question Jobs' leadership and his ability to execute on the company's strategic plans. The tensions between Jobs and the board had been building for some time, fueled by disagreements over product direction, resource allocation, and management style.\n\nIn August 1996, Apple's CEO, Gil Amelio, summoned Jobs to a meeting in San Francisco. Amelio had recently taken over

### Langchain-HypotheticalDocumentEmbedder

In [24]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder

from langchain_classic.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Step 1: Load and split documents
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

In [25]:
# Step 2: Set up LLM and embeddings

base_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

According to the official documentation and LangChain source code (mapping in PROMPT_MAP), the default options are:

- web_search
- sci_fact
- arguana
- trec_covid
- fiqa
- dbpedia_entity
- trec_news
- mr_tydi

![image.png](attachment:image.png)

In [26]:
hyde_embedding_function=HypotheticalDocumentEmbedder.from_llm(llm=llm,base_embeddings=base_embeddings,prompt_key="web_search")

In [27]:
# Step 4: Store documents in Chroma with HyDE embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=hyde_embedding_function,
    persist_directory="output/langchain"
)

In [28]:

# Step 5: RAG answer generation prompt
rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
rag_chain = create_stuff_documents_chain(llm=llm, prompt=rag_prompt)

In [29]:
def rag_pipeline(query):
    matched_docs=vectorstore.similarity_search(query=query,k=4)
    print(matched_docs)
    response=rag_chain.invoke({
        "input":query,
        "context":matched_docs
    })
    return response

In [31]:
query="What memory moduels does Langchain Provide"
answer=rag_pipeline(query)
print("Final Answer",answer)


[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using standard'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using standard'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or

In [37]:
from langchain_classic.prompts import  PromptTemplate
custom=PromptTemplate.from_template(
    "Generate a consise technical momo answering: {query}"
)

hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    custom_prompt=custom
)
